# What Is Biology? Life, Evolution, and Living Systems Workflow

This notebook scaffold supports growth modeling, Hardy-Weinberg expectations, biodiversity summaries, sequence comparison, biological levels, and provenance documentation.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

article_dir = Path.cwd().parent
levels = pd.read_csv(article_dir / 'data' / 'biological_levels.csv')
levels

In [ ]:
growth = pd.read_csv(article_dir / 'data' / 'growth_observations.csv')
rows = []
for scenario, group in growth.groupby('scenario'):
    slope, intercept = np.polyfit(group['time'], np.log(group['population']), 1)
    rows.append({'scenario': scenario, 'growth_rate': slope, 'N0': np.exp(intercept), 'doubling_time': np.log(2) / slope})
pd.DataFrame(rows).round(5)

In [ ]:
hw = pd.read_csv(article_dir / 'data' / 'hardy_weinberg_cases.csv')
hw['q'] = 1 - hw['allele_frequency_p']
hw['AA'] = hw['allele_frequency_p'] ** 2
hw['Aa'] = 2 * hw['allele_frequency_p'] * hw['q']
hw['aa'] = hw['q'] ** 2
hw.round(4)

In [ ]:
counts = pd.read_csv(article_dir / 'data' / 'biodiversity_counts.csv').set_index('site')
def shannon(x):
    p = x[x > 0] / x.sum()
    return float(-(p * np.log(p)).sum())
pd.DataFrame({'richness': (counts > 0).sum(axis=1), 'total_abundance': counts.sum(axis=1), 'shannon_diversity': counts.apply(shannon, axis=1)}).round(4)

In [ ]:
seqs = pd.read_csv(article_dir / 'data' / 'sequences.csv')
reference = seqs.loc[seqs['sequence_id'] == 'reference_B', 'sequence'].iloc[0]
seqs['similarity_to_reference_B'] = seqs['sequence'].apply(lambda s: 1 - sum(a != b for a, b in zip(s, reference)) / len(reference))
seqs.round(4)